In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt
import os
import random
from tqdm import tqdm
from torchvision import transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Dataset
from IPython import display

# 权重初始化函数
def init_weights_(m):
    if isinstance(m, nn.Conv2d):
        nn.init.xavier_normal_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d) or isinstance(m, nn.BatchNorm1d):
        nn.init.ones_(m.weight)
        nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

In [ ]:
# 数据集截取工具
class PartialDataset(Dataset):
    def __init__(self, dataset, n_items=10):
        self.dataset = dataset
        self.n_items = n_items

    def __getitem__(self, idx):
        return self.dataset.__getitem__(idx)

    def __len__(self):
        return min(self.n_items, len(self.dataset))

# CIFAR-10 DataLoader
def get_cifar_loader(root='./data/', batch_size=128, train=True, shuffle=True, num_workers=2, n_items=-1):
    normalize = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    data_transforms = transforms.Compose([transforms.ToTensor(), normalize])
    
    dataset = datasets.CIFAR10(root=root, train=train, download=True, transform=data_transforms)
    
    if n_items > 0:
        dataset = PartialDataset(dataset, n_items)

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers)
    return loader

In [ ]:
class VGG_A(nn.Module):
    """VGG_A model

    size of Linear layers is smaller since input assumed to be 32x32x3, instead of
    224x224x3
    """

    def __init__(self, inp_ch=3, num_classes=10, init_weights=True):
        super().__init__()

        self.features = nn.Sequential(
            # stage 1
            nn.Conv2d(in_channels=inp_ch, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # stage 2
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # stage 3
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1),
            nn.ReLU(True),
            nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding=1),
            nn.ReLU(True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # stage 4
            nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, padding=1),
            nn.ReLU(True),
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, padding=1),
            nn.ReLU(True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # stage5
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, padding=1),
            nn.ReLU(True),
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, padding=1),
            nn.ReLU(True),
            nn.MaxPool2d(kernel_size=2, stride=2))

        self.classifier = nn.Sequential(
            nn.Linear(512 * 1 * 1, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes))

        if init_weights:
            self._init_weights()

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x.view(-1, 512 * 1 * 1))
        return x

    def _init_weights(self):
        for m in self.modules():
            init_weights_(m)

class VGG_A_BatchNorm(nn.Module):
    """VGG_A 加上 Batch Normalization"""
    def __init__(self, inp_ch=3, num_classes=10, init_weights=True):
        super().__init__()

        self.features = nn.Sequential(
            # stage 1
            nn.Conv2d(in_channels=inp_ch, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), # 添加 BN 层
            nn.ReLU(True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # stage 2
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), # 添加 BN 层
            nn.ReLU(True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # stage 3
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # stage 4
            nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # stage5
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(512 * 1 * 1, 512),
            nn.BatchNorm1d(512), # 全连接层也可以加 BN
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

        if init_weights:
            self._init_weights()

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x.view(-1, 512 * 1 * 1))
        return x

    def _init_weights(self):
        for m in self.modules():
            init_weights_(m)

In [ ]:
# 确保使用 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def get_accuracy(model, data_loader, device):
    """计算模型在给定数据集上的准确率"""
    model.eval() # 设置为评估模式
    correct = 0
    total = 0
    with torch.no_grad(): # 不计算梯度，节省显存
        for x, y in data_loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            _, predicted = torch.max(outputs.data, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()
    return correct / total

def train(model, optimizer, criterion, train_loader, val_loader, epochs_n=20):
    model.to(device)
    learning_curve = [np.nan] * epochs_n
    
    batches_n = len(train_loader)
    losses_list = [] # 用于之后画 Loss Landscape

    for epoch in tqdm(range(epochs_n), unit='epoch'):
        model.train()
        loss_list = []  # 记录当前 epoch 每一步的 loss
        running_loss = 0.0

        for data in train_loader:
            x, y = data
            x, y = x.to(device), y.to(device)
            
            optimizer.zero_grad()
            prediction = model(x)
            loss = criterion(prediction, y)
            
            loss_list.append(loss.item())
            running_loss += loss.item()
            
            loss.backward()
            optimizer.step()

        losses_list.append(loss_list)
        
        # 记录每个 epoch 的平均 loss 用于画训练曲线
        learning_curve[epoch] = running_loss / batches_n
        
        # 计算验证集准确率
        val_acc = get_accuracy(model, val_loader, device)
        
        # 实时打印训练进度
        display.clear_output(wait=True)
        print(f"Epoch: {epoch+1}/{epochs_n} | Train Loss: {learning_curve[epoch]:.4f} | Val Accuracy: {val_acc*100:.2f}%")
        
        # 画出学习曲线
        plt.figure(figsize=(6, 4))
        plt.plot(learning_curve)
        plt.title('Training Loss Curve')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.show()

    return losses_list

In [ ]:
# Cell 5: 测试跑通训练流程
train_loader = get_cifar_loader(train=True)
val_loader = get_cifar_loader(train=False)

model_bn = VGG_A_BatchNorm()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_bn.parameters(), lr=0.001)

# 开始训练
losses_bn = train(model_bn, optimizer, criterion, train_loader, val_loader, epochs_n=5)

In [ ]:
# Cell 6: 测试无 BN 的标准 VGG-A 模型 (5 epochs)
print("--- 开始训练标准 VGG-A (无 BN) ---")
model_no_bn = VGG_A()
criterion = nn.CrossEntropyLoss()

# 保持控制变量：相同的优化器和学习率
optimizer_no_bn = torch.optim.Adam(model_no_bn.parameters(), lr=0.001)

# 开始训练
losses_no_bn = train(model_no_bn, optimizer_no_bn, criterion, train_loader, val_loader, epochs_n=5)

In [ ]:
# Cell 7: Loss Landscape 实验数据收集
import torch.optim as optim
import copy

def run_loss_landscape_exp(model_class, lr_list, max_steps=1500):
    base_model = model_class().to(device)
    initial_weights = copy.deepcopy(base_model.state_dict())
    
    all_losses = [] 
    criterion = nn.CrossEntropyLoss()
    
    for lr in lr_list:
        print(f"正在收集 {model_class.__name__} (学习率: {lr}) 的 Loss 数据...")
        model = model_class().to(device)
        model.load_state_dict(initial_weights)

        optimizer = optim.SGD(model.parameters(), lr=lr)
        
        step = 0
        lr_loss_history = []
        model.train()
        
        while step < max_steps:
            for inputs, labels in train_loader:
                if step >= max_steps: break
                inputs, labels = inputs.to(device), labels.to(device)
                
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                
                lr_loss_history.append(loss.item())
                step += 1
                
        all_losses.append(lr_loss_history)
        
    all_losses = np.array(all_losses) 
    # 计算同一个 step 下，不同学习率导致的 Loss 最大值和最小值
    return np.max(all_losses, axis=0), np.min(all_losses, axis=0)

# 设定不同学习率
learning_rates = [1e-3, 2e-3, 1e-4, 5e-4]
max_steps = 1500 

print("--- 1/2: 开始探测标准 VGG-A (无 BN) 的 Loss 地形 ---")
max_no_bn, min_no_bn = run_loss_landscape_exp(VGG_A, learning_rates, max_steps)

print("\n--- 2/2: 开始探测 VGG-A_BatchNorm 的 Loss 地形 ---")
max_bn, min_bn = run_loss_landscape_exp(VGG_A_BatchNorm, learning_rates, max_steps)
print("\n数据收集完毕，准备绘图！")

In [ ]:
# Cell 8: 绘制并保存 Loss Landscape 对比图
steps = np.arange(max_steps)

plt.figure(figsize=(12, 7))

# 绘制无 BN 的地形 (绿色区间)
plt.plot(steps, max_no_bn, color='darkgreen', alpha=0.2, linewidth=1)
plt.plot(steps, min_no_bn, color='darkgreen', alpha=0.2, linewidth=1)
plt.fill_between(steps, min_no_bn, max_no_bn, color='green', alpha=0.3, label='Standard VGG')

# 绘制有 BN 的地形 (红色区间)
plt.plot(steps, max_bn, color='darkred', alpha=0.2, linewidth=1)
plt.plot(steps, min_bn, color='darkred', alpha=0.2, linewidth=1)
plt.fill_between(steps, min_bn, max_bn, color='red', alpha=0.4, label='Standard VGG + BatchNorm')

plt.title('Loss Landscape: Standard VGG vs VGG + BatchNorm', fontsize=14)
plt.xlabel('Training Steps', fontsize=12)
plt.ylabel('Loss Value Variation', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)

plt.savefig('loss_landscape_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print(" Loss Landscape 对比图已生成并保存")

In [ ]:
# Cell 9: 引入更强的数据增强手段：随机裁剪和随机水平翻转
transform_augmented = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# 测试集保持不变，只做标准化
transform_standard = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])


trainset_aug = datasets.CIFAR10(root='./data/', train=True, download=True, transform=transform_augmented)
train_loader_aug = DataLoader(trainset_aug, batch_size=128, shuffle=True, num_workers=2)

testset = datasets.CIFAR10(root='./data/', train=False, download=True, transform=transform_standard)
test_loader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)


In [ ]:
# Cell 10:训练与调度函数
import torch.optim as optim

def train_high_accuracy(model_class, total_epochs=30, init_lr=0.01):
    print(f"\n--- 🚀 启动高准确率模型训练，预计训练 {total_epochs} 个 Epoch ---")
    model = model_class().to(device)
    criterion = nn.CrossEntropyLoss()
    
    # 使用带动量和 L2 正则化 (weight_decay) 的 SGD
    optimizer = optim.SGD(model.parameters(), lr=init_lr, momentum=0.9, weight_decay=5e-4)
    
    # 使用余弦退火学习率策略
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs)
    
    learning_curve = []
    
    for epoch in range(total_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in train_loader_aug:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
        # 学习率步进
        scheduler.step()
        
        epoch_loss = running_loss / len(train_loader_aug)
        epoch_acc = 100 * correct / total
        learning_curve.append(epoch_loss)
        
        print(f"Epoch [{epoch+1}/{total_epochs}] - Loss: {epoch_loss:.4f} - Train Acc: {epoch_acc:.2f}% - LR: {scheduler.get_last_lr()[0]:.6f}")
        
    return model, learning_curve

In [ ]:
# Cell 11: 执行训练并评估最终成绩
# 1. 开始训练 (30 个 epoch)
best_model, high_acc_loss_curve = train_high_accuracy(VGG_A_BatchNorm, total_epochs=30, init_lr=0.05)

# 2. 保存模型权重
torch.save(best_model.state_dict(), 'vgga_bn_high_acc.pth')
print("\n✅ 高级模型权重已保存为 'vgga_bn_high_acc.pth'")

# 3. 最终测试集评估
best_model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = best_model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

final_accuracy = 100 * correct / total

print("\n" + "="*40)
print(f"最终测试集准确率 (Final Accuracy): {final_accuracy:.2f}%")
print(f"最终测试集错误率 (Final Error): {100 - final_accuracy:.2f}%")
print("="*40)

In [ ]:
# 测试不同激活函数
print("--- 尝试不同的策略：使用 Tanh 激活函数代替 ReLU ---")

class VGG_A_Tanh(nn.Module):
    """用于对比实验：将基础 VGG-A 的 ReLU 替换为 Tanh"""
    def __init__(self, inp_ch=3, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(inp_ch, 64, 3, padding=1), nn.Tanh(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.Tanh(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.Tanh(), 
            nn.Conv2d(256, 256, 3, padding=1), nn.Tanh(), nn.MaxPool2d(2, 2),
            nn.Conv2d(256, 512, 3, padding=1), nn.Tanh(), 
            nn.Conv2d(512, 512, 3, padding=1), nn.Tanh(), nn.MaxPool2d(2, 2),
            nn.Conv2d(512, 512, 3, padding=1), nn.Tanh(), 
            nn.Conv2d(512, 512, 3, padding=1), nn.Tanh(), nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(512, 512), nn.Tanh(),
            nn.Linear(512, 512), nn.Tanh(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x.view(-1, 512))

model_tanh = VGG_A_Tanh()
optimizer_tanh = torch.optim.Adam(model_tanh.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# 只跑5个epoch看个趋势
losses_tanh = train(model_tanh, optimizer_tanh, criterion, train_loader, val_loader, epochs_n=5)

In [ ]:
# 卷积核可视化 (网络洞见)
import matplotlib.pyplot as plt
import numpy as np

best_model.eval()

# 提取特征提取层的第一层卷积核权重 (Conv2d)
# 形状应为 [64, 3, 3, 3] (out_channels, in_channels, kernel_size, kernel_size)
filters = best_model.features[0].weight.data.cpu().numpy()

fig, axes = plt.subplots(8, 8, figsize=(8, 8))
fig.suptitle('Visualization of 1st Layer Filters (VGG-A + BN)', fontsize=16)

for i, ax in enumerate(axes.flat):
    if i < 64:
        # 获取第 i 个卷积核
        f = filters[i]
        # 将通道维度移到最后变为 (3, 3, 3) 以适应 matplotlib
        f = np.transpose(f, (1, 2, 0))
        # 最小-最大归一化到 [0, 1] 以便显示RGB图像
        f_min, f_max = f.min(), f.max()
        f = (f - f_min) / (f_max - f_min + 1e-8)
        
        ax.imshow(f)
        ax.axis('off')

plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.savefig('filter_visualization.png', dpi=300)
plt.show()
print("卷积核可视化图已保存为 'filter_visualization.png'")

In [ ]:
# 测试不同数量的卷积核 (Filters)
print("--- 策略对比: 尝试减少卷积核数量 (VGG_A_Light) ---")

class VGG_A_Light(nn.Module):
    """通道数减半的轻量版 VGG-A"""
    def __init__(self, inp_ch=3, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(inp_ch, 32, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(True), 
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(True), 
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(True), 
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x.view(-1, 256))

model_light = VGG_A_Light()
optimizer_light = torch.optim.Adam(model_light.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# 3个 epoch 获取对比数据
losses_light = train(model_light, optimizer_light, criterion, train_loader, val_loader, epochs_n=3)

In [ ]:
# 测试不同的损失函数 (MultiMarginLoss)
print("--- 策略对比: 尝试不同的损失函数 (MultiMarginLoss) ---")

model_margin = VGG_A()
# 使用不同的损失函数 MultiMarginLoss
criterion_margin = nn.MultiMarginLoss()
# 保持 L2 正则化 (weight_decay)
optimizer_margin = torch.optim.Adam(model_margin.parameters(), lr=0.001, weight_decay=1e-4)

# 跑 3 个 epoch 获取对比数据
losses_margin = train(model_margin, optimizer_margin, criterion_margin, train_loader, val_loader, epochs_n=3)